In [26]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding
from tabpfn.constants import ModelVersion
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit,ShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from benchmark_datasets import OpenMLBenchmark
import warnings
warnings.filterwarnings("ignore")
load_dotenv(".env")

True

In [27]:
benchmarks = OpenMLBenchmark("classification")

In [28]:
import torch
import torch.nn as nn
from torch.func import functional_call


class MLP(nn.Module):
    """Small MLP student.

    A hidden layer + ReLU is required so the student can model non-linear targets;
    the original single-`Linear` version was just logistic regression and could not
    fit datasets like tic-tac-toe.
    """

    def __init__(self, input_size, output_size=1, hidden_size=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x):
        return self.mlp(x)


class ParameterMap(nn.Module):
    """Hypernetwork: maps a fixed TabPFN dataset embedding -> every weight of an MLP.

    Returns a ``{name: tensor}`` dict so the weights can be injected via
    ``torch.func.functional_call``. That keeps the hypernetwork in the autograd graph
    (unlike reassigning ``nn.Parameter`` objects, which silently detaches it and was
    why the original loop never trained anything).
    """

    def __init__(self, input_size, named_params):
        super().__init__()
        self.names, self.shapes = [], []
        self.parameter_maps = nn.ModuleList()
        for name, param in named_params:
            self.names.append(name)
            self.shapes.append(param.shape)
            self.parameter_maps.append(nn.Sequential(
                nn.Linear(input_size, param.numel()),
                nn.LayerNorm(param.numel()),
            ))

    def forward(self, x):
        return {
            name: pmap(x).reshape(shape)
            for name, pmap, shape in zip(self.names, self.parameter_maps, self.shapes)
        }


In [29]:
dataset = benchmarks.load("tic-tac-toe")
X,y = dataset.X,dataset.y

In [30]:
from sklearn.neural_network import MLPClassifier

In [31]:
from sklearn.metrics import accuracy_score, roc_auc_score


def to_t(a, dtype=torch.float32):
    return torch.as_tensor(np.asarray(a), dtype=dtype)


@torch.no_grad()
def accuracy(model, X_t, y):
    """Binary accuracy from raw logits (threshold at 0)."""
    pred = (model(X_t).squeeze(-1) > 0).int().numpy()
    return accuracy_score(y, pred)


@torch.no_grad()
def auc(model, X_t, y):
    """ROC AUC from raw logits (monotonic in probability, so the threshold is irrelevant)."""
    return roc_auc_score(y, model(X_t).squeeze(-1).numpy())


def accuracy_sklrn(model, X_t, y):
    """Binary accuracy from raw logits (threshold at 0)."""
    pred = (model.predict(X_t) > 0).astype(np.int16)
    return accuracy_score(y, pred)


def auc_sklrn(model, X_t, y):
    """ROC AUC from a scikit-learn classifier's positive-class probability."""
    return roc_auc_score(y, model.predict_proba(X_t)[:, 1])


def gradient_train(model, X_t, target, steps=200, lr=1e-3, params=None):
    """Plain gradient training with BCE-with-logits. `target` may be hard labels or soft probs."""
    opt = torch.optim.Adam(params if params is not None else model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        loss = bce(model(X_t).squeeze(-1), target)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


HIDDEN, HYPER_STEPS, FT_STEPS = 10, 1000, 300
AUGMENT, NOISE_STD = 4, 0.1   # synthetic noisy copies of the train set, soft-labeled by TabPFN
rng = np.random.default_rng(0)
# each model tracks both accuracy and ROC AUC across splits
results = {m: {"acc": [], "auc": []}
           for m in ["tabpfn", "distilled", "distilled+finetuned", "baseline_mlp"]}

for train_idx, test_idx in benchmarks.splits(X, y, train_size=64, task="classification"):
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # Standardize features (fit on train only). The MLP students train far better on scaled
    # inputs; TabPFN is scale-robust, so fitting it on the same scaled space is harmless and
    # keeps everything (teacher, embedding, augmentation, baseline) in one consistent space.
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    X_train_t, y_train_t = to_t(X_train), to_t(y_train)
    X_test_t = to_t(X_test)

    # --- Teacher: TabPFN ---
    tpfn = TabPFNClassifier().fit(X_train, y_train)
    results["tabpfn"]["acc"].append(accuracy_score(y_test, tpfn.predict(X_test)))
    results["tabpfn"]["auc"].append(roc_auc_score(y_test, tpfn.predict_proba(X_test)[:, 1]))
    soft_targets = to_t(tpfn.predict_proba(X_train)[:, 1])   # teacher probabilities (distillation target)

    # --- Fixed dataset embedding (constant for this dataset; computed once) ---
    embeddings = TabPFNEmbedding().fit_transform(X_train, y_train)
    dataset_embedding = to_t(embeddings.mean(axis=(0, 1)))   # (emb_dim,)

    # --- Hypernetwork distillation: student is the MLP whose weights the hypernet emits;
    #     train it to match the teacher's soft probabilities. functional_call keeps the
    #     hypernet in the graph so gradients reach it. ---
    mlp = MLP(X.shape[1], hidden_size=HIDDEN)
    hyper = ParameterMap(dataset_embedding.shape[0], list(mlp.named_parameters()))
    opt = torch.optim.AdamW(hyper.parameters(), lr=1e-3)
    bce = nn.BCEWithLogitsLoss()
    for _ in range(HYPER_STEPS):
        gen = hyper(dataset_embedding)
        logits = functional_call(mlp, gen, (X_train_t,)).squeeze(-1)
        loss = bce(logits, soft_targets)
        opt.zero_grad()
        loss.backward()
        opt.step()

    # bake the distilled weights into the MLP itself
    with torch.no_grad():
        gen = hyper(dataset_embedding)
        for name, p in mlp.named_parameters():
            p.copy_(gen[name])
    results["distilled"]["acc"].append(accuracy(mlp, X_test_t, y_test))
    results["distilled"]["auc"].append(auc(mlp, X_test_t, y_test))

    # --- Augment: noisy copies of the train rows, soft-labeled by the TabPFN teacher.
    #     Real rows keep their true labels; synthetic rows get the teacher's probability,
    #     so fine-tuning has more signal but stays anchored on ground truth. ---
    feat_std = X_train.std(axis=0, keepdims=True) + 1e-8
    X_syn = np.concatenate(
        [X_train + NOISE_STD * feat_std * rng.standard_normal(X_train.shape) for _ in range(AUGMENT)]
    ).astype(np.float32)
    X_ft_t = torch.cat([X_train_t, to_t(X_syn)])
    ft_targets = torch.cat([y_train_t, to_t(tpfn.predict_proba(X_syn)[:, 1])])

    # --- Fine-tune the distilled MLP on the augmented set (continues from distilled weights) ---
    gradient_train(mlp, X_ft_t, ft_targets, steps=FT_STEPS)
    results["distilled+finetuned"]["acc"].append(accuracy(mlp, X_test_t, y_test))
    results["distilled+finetuned"]["auc"].append(auc(mlp, X_test_t, y_test))

    # --- Baseline: identical-width MLP trained from random init on the real labels ---
    baseline = MLPClassifier(hidden_layer_sizes=(HIDDEN,), solver='adam')
    baseline.fit(X_train, y_train)
    results["baseline_mlp"]["acc"].append(accuracy_sklrn(baseline, X_test_t, y_test))
    results["baseline_mlp"]["auc"].append(auc_sklrn(baseline, X_test_t, y_test))

n_splits = len(results["tabpfn"]["acc"])
print(f"{'model':22s}  {'accuracy':18s} {'roc_auc':18s} (over {n_splits} splits)")
for name, d in results.items():
    acc_s = f"{np.mean(d['acc']):.3f} +/- {np.std(d['acc']):.3f}"
    auc_s = f"{np.mean(d['auc']):.3f} +/- {np.std(d['auc']):.3f}"
    print(f"{name:22s}  {acc_s:18s} {auc_s:18s}")


model                   accuracy           roc_auc            (over 5 splits)
tabpfn                  0.892 +/- 0.093    0.931 +/- 0.087   
distilled               0.660 +/- 0.018    0.672 +/- 0.018   
distilled+finetuned     0.608 +/- 0.013    0.616 +/- 0.042   
baseline_mlp            0.679 +/- 0.028    0.682 +/- 0.031   


## Evaluate every classification dataset

The single-dataset loop above is generalized below into a function that runs all four models
(TabPFN teacher, hypernetwork-distilled MLP, the same fine-tuned, and a from-scratch MLP baseline)
across **every** classification dataset in the benchmark suite, then aggregates the results.

The training is unified to softmax + (soft) cross-entropy so the **same code handles binary and
multiclass** datasets: the student MLP has `output_size = n_classes`, distillation matches TabPFN's
full `predict_proba` matrix, and ROC AUC is computed one-vs-rest (macro) for multiclass via
`benchmark_datasets._roc_auc`.

**Synthetic data augmentation for fine-tuning.** Fine-tuning is done on the real training rows
(kept with their true labels) *plus* `augment` noisy copies of them, where the noise is Gaussian
and scaled per-feature (`noise_std * feature_std`). Those synthetic rows have no ground-truth label,
so they are soft-labeled by the TabPFN teacher's `predict_proba`. This densifies the input space
around the few real points with teacher targets, giving the student more signal to fine-tune on
while staying anchored on the true labels (`_augment_with_teacher`).


In [39]:
import torch.nn.functional as F
from benchmark_datasets import _roc_auc   # binary positive-class / multiclass one-vs-rest macro AUC
from aug_dataset import aug_dataset

MODEL_NAMES = ["tabpfn", "distilled", "distilled+finetuned", "baseline_mlp"]


def _soft_cross_entropy(logits, soft_targets):
    """Distillation loss: cross-entropy against the teacher's full probability matrix.

    Works for binary and multiclass alike -- binary is just the two-column case.
    """
    return -(soft_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()


def _augment_with_teacher(X_train, y_train_onehot, teacher, *, augment, noise_std, rng):
    """Augment the fine-tuning set with synthetic, teacher-labeled examples.

    Each of ``augment`` passes copies the training rows and perturbs them with Gaussian noise
    scaled per-feature (``noise_std * feature_std``, so it is invariant to raw feature scale),
    then labels those synthetic rows with TabPFN's ``predict_proba``. The real rows keep their
    true (one-hot) labels, so fine-tuning stays anchored on ground truth while the synthetic
    rows densify the input space around it with teacher targets.

    Returns ``(X_ft_tensor, target_tensor)`` ready for ``_soft_cross_entropy``.
    """
    X_syn = to_t(aug_dataset(augment,X_train))
    #if augment <= 0:
    #    return X_train_t, y_train_onehot
    #feat_std = X_train.std(axis=0, keepdims=True) + 1e-8

    #X_syn = np.concatenate(
    #    [X_train + noise_std * feat_std * rng.standard_normal(X_train.shape) for _ in range(augment)]
    #).astype(np.float32)
    syn_soft = to_t(teacher.predict_proba(X_syn))
    #X_ft = torch.cat([X_train_t, to_t(X_syn)])
    #targets = torch.cat([y_train_onehot, syn_soft])
    return X_syn, syn_soft


@torch.no_grad()
def _mlp_acc(model, X_t, y):
    """Accuracy of the student MLP from its softmax logits (argmax over classes)."""
    return accuracy_score(y, model(X_t).argmax(dim=-1).numpy())


@torch.no_grad()
def _mlp_auc(model, X_t, y, classes):
    """ROC AUC of the student MLP from its softmax probabilities."""
    return _roc_auc(y, torch.softmax(model(X_t), dim=-1).numpy(), classes)


def evaluate_models_on_dataset(
    ds, *, train_size=8, n_splits=5, hidden=HIDDEN, hyper_steps=HYPER_STEPS, ft_steps=FT_STEPS,
    augment=2, noise_std=0.1,
):
    """Run the four models across repeated splits of a single ``LoadedDataset``.

    Generalizes the single-dataset loop above to any number of classes (softmax +
    cross-entropy throughout). Features are standardized per split (fit on train only); the
    MLP students train far better on scaled inputs and TabPFN is scale-robust, so the teacher,
    embedding, augmentation and baseline all share one consistent scaled space. Fine-tuning
    augments the real training rows with ``augment`` noisy copies (per-feature ``noise_std``)
    that are soft-labeled by the TabPFN teacher. Returns ``{model: {"acc": [...], "auc": [...]}}``
    with one entry per split.
    """
    X, y = ds.X, ds.y
    classes = np.unique(y)
    n_classes = len(classes)
    rng = np.random.default_rng(0)
    res = {m: {"acc": [], "auc": []} for m in MODEL_NAMES}

    for train_idx, test_idx in benchmarks.splits(
        X, y, task="classification", n_splits=n_splits, train_size=train_size
    ):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        # Standardize features (fit on train only) -- helps the MLP students; harmless to TabPFN.
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train).astype(np.float32)
        X_test = scaler.transform(X_test).astype(np.float32)

        X_train_t, X_test_t = to_t(X_train), to_t(X_test)
        y_train_long = to_t(y_train, dtype=torch.long)

        # --- Teacher: TabPFN ---
        tpfn = TabPFNClassifier().fit(X_train, y_train)
        res["tabpfn"]["acc"].append(accuracy_score(y_test, tpfn.predict(X_test)))
        res["tabpfn"]["auc"].append(_roc_auc(y_test, tpfn.predict_proba(X_test), classes))
        soft_targets = to_t(tpfn.predict_proba(X_train))            # (n_train, n_classes)

        # --- Fixed dataset embedding ---
        embeddings = TabPFNEmbedding().fit_transform(X_train, y_train)
        dataset_embedding = to_t(embeddings.mean(axis=(0, 1)))

        # --- Hypernetwork distillation onto the teacher's probabilities ---
        mlp = MLP(X.shape[1], output_size=n_classes, hidden_size=hidden)
        hyper = ParameterMap(dataset_embedding.shape[0], list(mlp.named_parameters()))
        opt = torch.optim.AdamW(hyper.parameters(), lr=1e-3)
        for _ in range(hyper_steps):
            gen = hyper(dataset_embedding)
            logits = functional_call(mlp, gen, (X_train_t,))
            loss = _soft_cross_entropy(logits, soft_targets)
            opt.zero_grad()
            loss.backward()
            opt.step()
        with torch.no_grad():
            gen = hyper(dataset_embedding)
            for name, p in mlp.named_parameters():
                p.copy_(gen[name])
        res["distilled"]["acc"].append(_mlp_acc(mlp, X_test_t, y_test))
        res["distilled"]["auc"].append(_mlp_auc(mlp, X_test_t, y_test, classes))

        # --- Fine-tune: real rows (true labels) + TabPFN-labeled synthetic noisy rows ---
        y_train_onehot = F.one_hot(y_train_long, n_classes).float()
        X_ft_t, ft_targets = _augment_with_teacher(
            X_train, y_train_onehot, tpfn, augment=augment, noise_std=noise_std, rng=rng
        )
        opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
        for _ in range(ft_steps):
            loss = _soft_cross_entropy(mlp(X_ft_t), ft_targets)
            opt.zero_grad()
            loss.backward()
            opt.step()
        res["distilled+finetuned"]["acc"].append(_mlp_acc(mlp, X_test_t, y_test))
        res["distilled+finetuned"]["auc"].append(_mlp_auc(mlp, X_test_t, y_test, classes))

        # --- Baseline: identical-width MLP trained from random init on the real labels ---
        baseline = MLPClassifier(hidden_layer_sizes=(hidden,), solver="adam")
        baseline.fit(X_train, y_train)
        res["baseline_mlp"]["acc"].append(accuracy_score(y_test, baseline.predict(X_test)))
        res["baseline_mlp"]["auc"].append(_roc_auc(y_test, baseline.predict_proba(X_test), classes))

    return res


def evaluate_all_classification(datasets=None, *, verbose=True, **kwargs):
    """Evaluate the four models over every classification dataset and aggregate.

    Args:
        datasets: optional subset of dataset names (defaults to the whole suite).
        verbose: print a per-dataset line as each finishes.
        **kwargs: forwarded to ``evaluate_models_on_dataset`` (``train_size``, ``n_splits``,
            ``hidden``, ``hyper_steps``, ``ft_steps``, ``augment``, ``noise_std``).

    Returns:
        ``(per_dataset, summary)`` DataFrames. ``per_dataset`` has acc/auc columns per model
        (averaged over splits); ``summary`` averages each model across all datasets.
    """
    specs = benchmarks.specs
    if datasets is not None:
        wanted = set(datasets)
        specs = [s for s in specs if s.name in wanted]

    rows = []
    for spec in specs:
        ds = benchmarks.load(spec)
        try:
            res = evaluate_models_on_dataset(ds, **kwargs)
        except Exception as exc:                       # keep the sweep going on a bad dataset
            print(f"[skip] {spec.name}: {exc}")
            continue
        row = {"dataset": spec.name, "n_rows": spec.n_rows, "n_classes": ds.n_classes}
        for m in MODEL_NAMES:
            row[f"{m}_acc"] = float(np.mean(res[m]["acc"]))
            row[f"{m}_auc"] = float(np.nanmean(res[m]["auc"]))
        rows.append(row)
        if verbose:
            cells = " | ".join(
                f"{m}={row[f'{m}_acc']:.3f}/{row[f'{m}_auc']:.3f}" for m in MODEL_NAMES
            )
            print(f"{spec.name:24s} (k={ds.n_classes}) {cells}")

    per_dataset = pd.DataFrame(rows)
    summary = pd.DataFrame(
        {
            "model": MODEL_NAMES,
            "acc_mean": [per_dataset[f"{m}_acc"].mean() for m in MODEL_NAMES],
            "acc_std": [per_dataset[f"{m}_acc"].std() for m in MODEL_NAMES],
            "auc_mean": [per_dataset[f"{m}_auc"].mean() for m in MODEL_NAMES],
            "auc_std": [per_dataset[f"{m}_auc"].std() for m in MODEL_NAMES],
        }
    )
    return per_dataset, summary


In [40]:
# Run the full sweep over every classification dataset, then aggregate.
# Heavy: ~50 datasets x n_splits x (TabPFN fit + embedding + hypernet + fine-tune + baseline).
# Pass e.g. datasets=["sonar", "wine"] or a smaller n_splits/hyper_steps for a quick smoke test.
per_dataset, summary = evaluate_all_classification()

print("\n=== Aggregate across all classification datasets ===")
display(summary.round(3))
display(per_dataset.round(3))

# Mean ROC AUC across all datasets for the teacher, student, and baseline.
auc_means = summary.set_index("model")["auc_mean"]
print("\n=== Mean ROC AUC across all classification datasets ===")
print(f"teacher  (tabpfn)              : {auc_means['tabpfn']:.4f}")
print(f"student  (distilled+finetuned) : {auc_means['distilled+finetuned']:.4f}")
print(f"baseline (baseline_mlp)        : {auc_means['baseline_mlp']:.4f}")


tic-tac-toe              (k=2) tabpfn=0.569/0.536 | distilled=0.519/0.515 | distilled+finetuned=0.535/0.527 | baseline_mlp=0.544/0.533
monks-problems-2         (k=2) tabpfn=0.626/0.496 | distilled=0.561/0.507 | distilled+finetuned=0.566/0.508 | baseline_mlp=0.548/0.501
sonar                    (k=2) tabpfn=0.634/0.695 | distilled=0.553/0.595 | distilled+finetuned=0.543/0.568 | baseline_mlp=0.587/0.650
ionosphere               (k=2) tabpfn=0.735/0.814 | distilled=0.644/0.636 | distilled+finetuned=0.606/0.575 | baseline_mlp=0.721/0.728
vehicle                  (k=4) tabpfn=0.446/0.730 | distilled=0.371/0.614 | distilled+finetuned=0.340/0.577 | baseline_mlp=0.386/0.638
wdbc                     (k=2) tabpfn=0.919/0.979 | distilled=0.755/0.842 | distilled+finetuned=0.733/0.824 | baseline_mlp=0.899/0.960
diabetes                 (k=2) tabpfn=0.663/0.731 | distilled=0.614/0.612 | distilled+finetuned=0.580/0.602 | baseline_mlp=0.603/0.656
ilpd                     (k=2) tabpfn=0.707/0.592 | dis

,model,acc_mean,acc_std,auc_mean,auc_std
0,tabpfn,0.735,0.153,0.711,0.157
1,distilled,0.634,0.132,0.606,0.108
2,distilled+finetuned,0.626,0.135,0.596,0.097
3,baseline_mlp,0.675,0.147,0.684,0.149


,dataset,n_rows,n_classes,tabpfn_acc,tabpfn_auc,distilled_acc,distilled_auc,distilled+finetuned_acc,distilled+finetuned_auc,baseline_mlp_acc,baseline_mlp_auc
0,tic-tac-toe,958,2,0.569,0.536,0.519,0.515,0.535,0.527,0.544,0.533
1,monks-problems-2,601,2,0.626,0.496,0.561,0.507,0.566,0.508,0.548,0.501
2,sonar,208,2,0.634,0.695,0.553,0.595,0.543,0.568,0.587,0.650
3,ionosphere,351,2,0.735,0.814,0.644,0.636,0.606,0.575,0.721,0.728
4,vehicle,846,4,0.446,0.730,0.371,0.614,0.340,0.577,0.386,0.638
5,wdbc,569,2,0.919,0.979,0.755,0.842,0.733,0.824,0.899,0.960
6,diabetes,768,2,0.663,0.731,0.614,0.612,0.580,0.602,0.603,0.656
7,ilpd,583,2,0.707,0.592,0.608,0.576,0.614,0.575,0.594,0.600
8,blood-transfusion,748,2,0.750,0.601,0.609,0.570,0.599,0.593,0.712,0.632
9,heart-statlog,270,2,0.707,0.802,0.661,0.707,0.640,0.687,0.682,0.794



=== Mean ROC AUC across all classification datasets ===
teacher  (tabpfn)              : 0.7111
student  (distilled+finetuned) : 0.5963
baseline (baseline_mlp)        : 0.6837


In [41]:
summary.to_csv("results/mlp_distillation_8_sum.csv")
per_dataset.to_csv("results/mlp_distillation_8_dataset.csv")